# Human gastric cancer decoder

Train a gene expression decoder using the saved scanVI input.


## Settings


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "experiments").is_dir() and (path / "lrpairs").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

from _shared.runtime import training_epochs, training_routes, scanvi_train_kwargs
from pathlib import Path
import sys

DATASET = "GP1"
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / "human_gastric_cancer"
REPO_ROOT = EXPERIMENT_DIR.parents[1]

import anndata as ad
import numpy as np
import yaml

from stvirtual.decoder.config import TrainConfig
from stvirtual.decoder.train import train_decoder



In [3]:
with (EXPERIMENT_DIR / "config.yaml").open(encoding="utf-8") as handle:
    experiment_config = yaml.safe_load(handle)

def resolve_experiment_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

def decoder_checkpoint(src, tgt):
    pattern = experiment_config["decoder_checkpoint"]
    return resolve_experiment_path(pattern.format(src=src, tgt=tgt))

ROUTES = [('normal', 'cancer')]
DATA_PATH = resolve_experiment_path(experiment_config["scanvi_dir"]) / "adata.h5ad"
DECODER_OUTPUT_DIR = decoder_checkpoint(*ROUTES[0]).parents[1]
LATENT_KEY = experiment_config.get("latent_key", "X_scanVI")
COUNTS_KEY = "counts"
SAMPLE_KEY = "status"
DEVICE = "cuda:0"
SEED = 2025

print(f"Input AnnData: {DATA_PATH}")
print("Decoder routes:", ROUTES)
for src, tgt in ROUTES:
    print(f"  {src} -> {tgt}: {decoder_checkpoint(src, tgt)}")


Input AnnData: /home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/scanvi/adata.h5ad
Decoder routes: [('normal', 'cancer')]
  normal -> cancer: /home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/decoder/checkpoints/normal_cancer.pt


In [4]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run preprocess.ipynb first to create the canonical AnnData object."
    )

adata = ad.read_h5ad(DATA_PATH, backed="r")
try:
    if LATENT_KEY not in adata.obsm:
        raise KeyError(f"adata.obsm does not contain {LATENT_KEY!r}")
    if COUNTS_KEY not in adata.layers:
        raise KeyError(f"adata.layers does not contain {COUNTS_KEY!r}")
    if SAMPLE_KEY not in adata.obs:
        raise KeyError(f"adata.obs does not contain {SAMPLE_KEY!r}")
    sample_names = list(dict.fromkeys(adata.obs[SAMPLE_KEY].astype(str)))
    missing_samples = sorted({sample for route in ROUTES for sample in route} - set(sample_names))
    if missing_samples:
        raise ValueError(f"Decoder routes reference missing samples: {missing_samples}")
    summary = {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "latent_shape": tuple(adata.obsm[LATENT_KEY].shape),
        "available_samples": sample_names,
        "routes": ROUTES,
    }
finally:
    adata.file.close()

summary


{'n_cells': 391,
 'n_genes': 36601,
 'latent_shape': (391, 10),
 'available_samples': ['normal', 'cancer'],
 'routes': [('normal', 'cancer')]}

In [5]:
TRAINING_KWARGS = dict(
    data_path=DATA_PATH,
    output_dir=DECODER_OUTPUT_DIR,
    latent_key=LATENT_KEY,
    counts_key=COUNTS_KEY,
    sample_key=SAMPLE_KEY,
    simulation_normalization_checkpoint=None,
    device=DEVICE,
    epochs=training_epochs(100),
    patience=20,
    batch_size=256,
    learning_rate=3e-4,
    seed=SEED,
    overwrite=False,
)
TRAINING_KWARGS


{'data_path': PosixPath('/home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/scanvi/adata.h5ad'),
 'output_dir': PosixPath('/home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/decoder'),
 'latent_key': 'X_scanVI',
 'counts_key': 'counts',
 'sample_key': 'status',
 'simulation_normalization_checkpoint': None,
 'device': 'cuda:0',
 'epochs': 100,
 'patience': 20,
 'batch_size': 256,
 'learning_rate': 0.0003,
 'seed': 2025,
 'overwrite': False}

## Train the decoder


In [6]:
checkpoints = {}
for src, tgt in ROUTES:
    decoder_config = TrainConfig(
        **TRAINING_KWARGS,
        sample_times={src: 0.0, tgt: 1.0},
        checkpoint_name=f"{src}_{tgt}.pt",
    )
    checkpoint = train_decoder(decoder_config).resolve()
    expected = decoder_checkpoint(src, tgt).resolve()
    if checkpoint != expected:
        raise RuntimeError(f"Decoder output {checkpoint} does not match config path {expected}")
    checkpoints[f"{src}_to_{tgt}"] = checkpoint

checkpoints


Train decoder:   0%|          | 0/100 [00:00<?, ?epoch/s]

{'normal_to_cancer': PosixPath('/home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/decoder/checkpoints/normal_cancer.pt')}

In [7]:
import torch

audits = {}
for route, checkpoint in checkpoints.items():
    payload = torch.load(checkpoint, map_location="cpu", weights_only=False)
    audit = {
        "checkpoint": str(checkpoint),
        "time_conditioning": payload.get("time_conditioning"),
        "latent_key": payload.get("latent_key"),
        "counts_key": payload.get("counts_key"),
        "n_genes": len(payload.get("gene_names", [])),
        "best_validation_loss": payload.get("best_validation_loss"),
    }
    if audit["time_conditioning"] is not False:
        raise ValueError("The public decoder must be trained without time conditioning")
    audits[route] = audit

audits


{'normal_to_cancer': {'checkpoint': '/home/zhouyj/stVirtual_tutorial/experiments/human_gastric_cancer/artifacts/checkpoints/decoder/checkpoints/normal_cancer.pt',
  'time_conditioning': False,
  'latent_key': 'X_scanVI',
  'counts_key': 'counts',
  'n_genes': 36601,
  'best_validation_loss': 0.3783140921777533}}